# Import Libraries

In [17]:
import pandas as pd
import re
import nltk
import numpy as np
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer

# Initialize

In this preprocessing step, we intentionally omitted aggressive text-cleaning techniques such as stopword removal and lemmatization. This approach was chosen to preserve the natural syntactic structure and contextual nuances of the sentences, which are crucial for the Bidirectional Attention Mechanism in Transformer models (DistilBERT and BERT-Base). Furthermore, retaining the original linguistic form ensures a fair, apples-to-apples comparative analysis across all four models, evaluating their innate capability to process raw human language.

# Preprocess Function

In [18]:
def preprocess_statement(statement):
    # Lowercase & Format String
    sentence = str(statement).lower()

    # Remove Tag Reddit & URL
    sentence = re.sub(r'\[.*?\]', '', sentence) 
    sentence = re.sub(r'http\S+|www\.\S+', '', sentence) 
    sentence = re.sub(r'\@\w+', '', sentence)

    # Remove symbols
    sentence = re.sub(r'[^a-zA-Z0-9\s.,!?\']', '', sentence)

    # Remove double space if any
    sentence = re.sub(r'\s+', ' ', sentence).strip()

    return sentence

# Main Preprocess

## GoEmotions Dataset

In [19]:
df_emo = pd.read_csv('../data/goemotions_kasar_pure.csv') # Load Dataset
df_emo['clean_text'] = df_emo['text'].apply(preprocess_statement) # Apply

In [20]:
df_emo['clean_text'] = df_emo['clean_text'].replace('', np.nan)
df_emo.isna().sum()

text          0
emotion       0
clean_text    2
dtype: int64

In [21]:
df_emo[df_emo.isna().any(axis=1)]

,text,emotion,clean_text
5873,[Facebook laugh react],Joy,NaN
8636,[NAME],Anger,NaN


It turns out that the empty values in the clean_text column come from unusable text entries, as the original text does not contain meaningful information for training the model. Therefore, we can continue and drop the unuseful columns

In [22]:
df_emo = df_emo.dropna(subset=['clean_text'])
df_emo = df_emo.reset_index(drop=True)
print(df_emo.isna().sum())

text          0
emotion       0
clean_text    0
dtype: int64


In [23]:
# Save the preprocessed GoEmotions Dataset into a new CSV
df_emo.to_csv('../data/goemotions_train.csv', index=False)

## IMDB Dataset

In [24]:
df_film = pd.read_csv('../data/film_kasar.csv')

# Make sure no null overview -> Null Overview means no use for the model
df_film = df_film.dropna(subset=['Overview'])

df_film['clean_overview'] = df_film['Overview'].apply(preprocess_statement)

In [25]:
df_film['clean_overview'] = df_film['clean_overview'].replace('', np.nan)
df_film.isna().sum()

Series_Title      0
Genre             0
Overview          0
clean_overview    0
dtype: int64

On the other hand, IMDB Dataset doesn't have any null columns even after preprocessing. So we can straight up save the dataset.

In [29]:
df_film.to_csv('../data/imdb_train.csv', index=False)

In [30]:
df_emo.head(5)

,text,emotion,clean_text
0,That game hurt.,Sadness,that game hurt.
1,Man I love reddit.,Love,man i love reddit.
2,So happy for [NAME]. So sad he's not here. Ima...,Joy,so happy for . so sad he's not here. imagine t...
3,"I just came home, what the fuck is this lineup...",Love,"i just came home, what the fuck is this lineup..."
4,By far the coolest thing I've seen on this thr...,Joy,by far the coolest thing i've seen on this thr...


In [31]:
df_film.head(5)

,Series_Title,Genre,Overview,clean_overview
0,The Shawshank Redemption,Drama,Two imprisoned men bond over a number of years...,two imprisoned men bond over a number of years...
1,The Godfather,"Crime, Drama",An organized crime dynasty's aging patriarch t...,an organized crime dynasty's aging patriarch t...
2,The Dark Knight,"Action, Crime, Drama",When the menace known as the Joker wreaks havo...,when the menace known as the joker wreaks havo...
3,The Godfather: Part II,"Crime, Drama",The early life and career of Vito Corleone in ...,the early life and career of vito corleone in ...
4,12 Angry Men,"Crime, Drama",A jury holdout attempts to prevent a miscarria...,a jury holdout attempts to prevent a miscarria...
